In [ ]:
import pickle
with open('/content/drive/MyDrive/final_test_df','rb') as f:
  test_data=pickle.load(f)

In [ ]:
import torch
device='cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
source_sentences_test=[]
target_sentences_test=[]
id_val=[]

In [ ]:
for language_pair, language_data in test_data.items():
    if(language_pair == "English-Hindi"):
      print(f"Language Pair: {language_pair}")
      for data_type, data_entries in language_data.items():
          print(f"  Data Type: {data_type}")
          for entry_id, entry_data in data_entries.items():
              source = entry_data["source"]
              target = entry_data["target"]
              if (data_type == "Test"):
                source_sentences_test.append(source)
                target_sentences_test.append(target)
                id_val.append(entry_id)

In [ ]:
test_x={'English':source_sentences_test,'Hindi':target_sentences_test}

In [ ]:
import pandas as pd
test_df=pd.DataFrame(test_x)

In [ ]:
with open('/content/drive/MyDrive/english_train_tokens','rb') as f:
  english_tokens_1=pickle.load(f)

with open('/content/drive/MyDrive/english_test_tokens','rb') as f:
  english_test_1=pickle.load(f)

In [ ]:
with open('/content/drive/MyDrive/hindi_train_tokens','rb') as f:
  hindi_tokens_1=pickle.load(f)

In [ ]:
en_train=english_tokens_1
en_test=english_test_1
hi_train=hindi_tokens_1

In [ ]:
def build_vocab(datasets, specials=["<PAD>", "<SOS>", "<EOS>", "<UNK>"]):
    vocab = set()
    for ds in datasets:
        for sent in ds:
            vocab.update(sent)

    vocab -= set(specials)
    idx2word = specials + sorted(list(vocab))
    word2idx = {w: i for i, w in enumerate(idx2word)}
    return idx2word, word2idx


In [ ]:

en_index2word, en_word2index = build_vocab([en_train, en_test])

hi_index2word, hi_word2index = build_vocab([hi_train])


In [ ]:
def encode_and_pad_fixed(sentences, word2index, max_length, device):

    encoded_tensors = []

    for sent in sentences:
        encoded = [word2index["<SOS>"]] + \
                  [word2index.get(word, word2index["<UNK>"]) for word in sent] + \
                  [word2index["<EOS>"]]

        if len(encoded) > max_length:
            encoded = encoded[:max_length]
        else:
            encoded += [word2index["<PAD>"]] * (max_length - len(encoded))

        encoded_tensors.append(torch.tensor(encoded, dtype=torch.long))
    padded = torch.stack(encoded_tensors)
    return padded.to(device)

max_len = 50  # manually chosen
en_test_tensor = encode_and_pad_fixed(english_test_1, en_word2index, max_len, device)


print("English test shape:", en_test_tensor.shape)


In [ ]:
import numpy as np
def load_glove_embeddings(glove_path):
    embeddings_index = {}
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print(f"Loaded {len(embeddings_index)} word vectors from GloVe.")
    return embeddings_index

glove_path = '/content/drive/MyDrive/glove.6B.300d.txt'
glove_embeddings = load_glove_embeddings(glove_path)


In [ ]:
import torch

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import json


class TestDataset(Dataset):
    def __init__(self, src_data):
        self.src_data = src_data

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return torch.tensor(self.src_data[idx], dtype=torch.long)


def collate_fn_test(batch, pad_id):
    max_len = max(len(x) for x in batch)
    padded = [torch.cat([x, torch.full((max_len - len(x),), pad_id,dtype=torch.long, device=x.device)]) for x in batch]
    return torch.stack(padded, dim=0)


In [ ]:

def build_embedding_matrix(word2idx, pretrained_source, embedding_dim=300, is_fasttext=False):
    vocab_size = len(word2idx)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))


    limit = np.sqrt(6 / embedding_dim)

    for word, idx in word2idx.items():
        try:
            if is_fasttext:
                embedding_matrix[idx] = pretrained_source.get_word_vector(word)
            else:
                embedding_matrix[idx] = pretrained_source[word]
        except (KeyError, AttributeError):

            embedding_matrix[idx] = np.random.uniform(-limit, limit, embedding_dim)

    return torch.tensor(embedding_matrix, dtype=torch.float32)


In [ ]:
save_path_hi = './fastest_hindi_embeddings.pt'
fas_hi_embeddings = torch.load(save_path_hi)

In [ ]:
src_embeddings = build_embedding_matrix(
    en_word2index, glove_embeddings, embedding_dim=300, is_fasttext=False
)
tgt_embeddings = build_embedding_matrix(
    hi_word2index, fas_hi_embeddings, embedding_dim=300, is_fasttext=True
)

In [ ]:
from code import TransformerMT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pad_id_src = en_word2index["<PAD>"]
pad_id_tgt = hi_word2index["<PAD>"]

model = TransformerMT(
    src_vocab_size=len(en_word2index),
    tgt_vocab_size=len(hi_word2index),
    src_embeddings=src_embeddings,
    tgt_embeddings=tgt_embeddings,
    src_pad_id=pad_id_src,
    tgt_pad_id=pad_id_tgt
).to(device)

checkpoint_path = "/content/drive/MyDrive/frf_transformer_epoch10.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print(f"Loaded checkpoint: {checkpoint_path}")


def greedy_decode(model, src, max_len=60, pad_id=0, device="cpu"):
    model.eval()
    batch_size = src.size(0)


    sos_id = 1
    tgt = torch.full((batch_size, 1), sos_id, dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_len - 1):
            logits = model(src, tgt)   # (B, T, vocab_size)
            next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B, 1)
            tgt = torch.cat([tgt, next_token], dim=1)

    return tgt  # (B, max_len)


test_dataset = TestDataset(en_test_tensor)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_fn_test(b, pad_id_src)
)

save_path = "/content/drive/MyDrive/nnn_est_predictions.json"

all_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Decoding", unit="batch"):
        src_ids = batch.to(device)


        batch_preds = greedy_decode(
            model, src_ids, max_len=60, pad_id=pad_id_tgt, device=device
        )


        batch_preds = [seq.tolist() for seq in batch_preds]
        all_preds.extend(batch_preds)


with open(save_path, "w", encoding="utf-8") as f:
    json.dump(all_preds, f, ensure_ascii=False, indent=2)

print(f"Saved test predictions to {save_path}")


In [ ]:
import json
with open("/content/drive/MyDrive/nnn_est_predictions.json", "r", encoding="utf-8") as f:
    preds = json.load(f)


In [ ]:
# !pip install sacrebleu rouge-score

In [ ]:
# !pip install rouge

In [ ]:
import json
from sacrebleu.metrics import BLEU, CHRF
from rouge import Rouge

# Load raw token-ID predictions
# with open("/content/drive/MyDrive/nnn_est_predictions.json", "r", encoding="utf-8") as f:
#     preds = json.load(f)

# Decode token IDs → Hindi strings
eos_id = hi_word2index["<EOS>"]
pad_id = hi_word2index["<PAD>"]
sos_id = hi_word2index["<SOS>"]

skip_ids = {sos_id, eos_id, pad_id}

decoded_preds = []
for seq in preds:
    tokens = []
    for tok_id in seq:
        if tok_id == eos_id:
            break
        if tok_id not in skip_ids:
            tokens.append(hi_index2word[tok_id])
    decoded_preds.append(" ".join(tokens))

# Clean reference strings
decoded_refs = [
    r[0] if isinstance(r, list) else r
    for r in test_df['Hindi']
]

# Compute metrics
refs_nested = [decoded_refs]

bleu_metric = BLEU()
chrf_metric = CHRF()

bleu = bleu_metric.corpus_score(decoded_preds, refs_nested)
chrf = chrf_metric.corpus_score(decoded_preds, refs_nested)

rouge = Rouge()
scores = rouge.get_scores(decoded_preds, decoded_refs, avg=True)

metrics = {
    "BLEU": bleu.score,
    "chrF": chrf.score,
    "ROUGE-1": scores["rouge-1"]["f"] * 100,
    "ROUGE-2": scores["rouge-2"]["f"] * 100,
    "ROUGE-L": scores["rouge-l"]["f"] * 100
}

print(metrics) # {'BLEU': 9.448198355791833, 'chrF': 31.478134674849905, 'ROUGE-1': 40.91412090337956, 'ROUGE-2': 15.23398518260228, 'ROUGE-L': 35.31283279789}